# Flooring Dataset — Complete Analysis & Model Preparation

**Goal:** clean, validate, analyze, engineer features, detect leakage, create quantity/cost logic, build a simple value-for-money score, prepare an ML-ready dataset, and export the final files.

> **Important:** `Quality_Level` is treated as a product attribute. It is **not** generated from `Price_EGP`, so the notebook avoids target leakage.


## 1. Import Libraries

In [1]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


## 2. Load Dataset

In [2]:
df = pd.read_csv('flooring.csv')
df.head()

,Brand,Product_Name,Floor_Type,Subcategory,Grade,Size_cm,Application,Quality_Level,Quality_Score,Unit,Waste_Factor,Quantity_Rule,Rule_Value,Required_For,Optional,Price_EGP,Category
0,Lecico,Porcelain 60x120,Porcelain,Floor Tile,First,60x120,Kitchen,Medium,8.7,m2,0.12,Area_m2,1,Reception,False,701,Flooring
1,Royal Ceramica,Porcelain 60x120,Porcelain,Floor Tile,First,60x120,Kitchen,High,8.7,m2,0.12,Area_m2,1,Reception,False,747,Flooring
2,Alfa Ceramic,Ceramic 60x60,Ceramic,Floor Tile,Second,60x60,Bathroom,Low,7.5,m2,0.10,Area_m2,1,Bedrooms,False,247,Flooring
3,Ceramica Cleopatra,Ceramic 60x60,Ceramic,Floor Tile,Third,60x60,Kitchen,High,7.5,m2,0.10,Area_m2,1,Bedrooms,False,238,Flooring
4,Alfa Ceramic,Ceramic 60x60,Ceramic,Floor Tile,First,60x60,Kitchen,Medium,7.5,m2,0.10,Area_m2,1,Bedrooms,False,354,Flooring


## 3. Initial Inspection

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1011 entries, 0 to 1010
Data columns (total 17 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Brand          1011 non-null   object 
 1   Product_Name   1011 non-null   object 
 2   Floor_Type     1011 non-null   object 
 3   Subcategory    1011 non-null   object 
 4   Grade          1011 non-null   object 
 5   Size_cm        1011 non-null   object 
 6   Application    1011 non-null   object 
 7   Quality_Level  1011 non-null   object 
 8   Quality_Score  1011 non-null   float64
 9   Unit           1011 non-null   object 
 10  Waste_Factor   1011 non-null   float64
 11  Quantity_Rule  1011 non-null   object 
 12  Rule_Value     1011 non-null   int64  
 13  Required_For   1011 non-null   object 
 14  Optional       1011 non-null   bool   
 15  Price_EGP      1011 non-null   int64  
 16  Category       1011 non-null   object 
dtypes: bool(1), float64(2), int64(2), object(12)
memory 

In [4]:
df.isna().sum()

,0
Brand,0
Product_Name,0
Floor_Type,0
Subcategory,0
Grade,0
Size_cm,0
Application,0
Quality_Level,0
Quality_Score,0
Unit,0


In [5]:
df.describe()

,Quality_Score,Waste_Factor,Rule_Value,Price_EGP
count,1011.000000,1011.000000,1011.0,1011.000000
mean,8.254896,0.112582,1.0,532.979228
std,0.579948,0.009666,0.0,232.111079
min,7.500000,0.100000,1.0,169.000000
25%,7.500000,0.100000,1.0,286.000000
50%,8.700000,0.120000,1.0,547.000000
75%,8.700000,0.120000,1.0,737.000000
max,8.700000,0.120000,1.0,947.000000


In [6]:
df.duplicated().sum()

np.int64(7)

## 4. Data Quality Checks

In [7]:
df.nunique()

,0
Brand,5
Product_Name,2
Floor_Type,2
Subcategory,1
Grade,3
Size_cm,2
Application,4
Quality_Level,3
Quality_Score,2
Unit,1


In [8]:
df[df['Price_EGP'] <= 0 ]

,Brand,Product_Name,Floor_Type,Subcategory,Grade,Size_cm,Application,Quality_Level,Quality_Score,Unit,Waste_Factor,Quantity_Rule,Rule_Value,Required_For,Optional,Price_EGP,Category


## 5. Cleaning

In [10]:
df_clean = df.copy()

text_cols = df_clean.select_dtypes(include="object").columns
for col in text_cols:
    df_clean[col] = df_clean[col].astype("string").str.strip()

In [11]:
df_clean = df_clean.drop_duplicates().reset_index(drop=True)

## 6. Check Existing Quality Information — No Price Leakage

In [ ]:
quality_counts = df_clean["Quality_Level"].value_counts(dropna=False)
quality_price_stats = df_clean.groupby("Quality_Level")["Price_EGP"].agg(
    ["count", "min", "median", "mean", "max"]
).sort_values("mean", ascending=False)


## 7. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(df_clean["Price_EGP"], bins=30)
plt.title("Flooring Price Distribution")
plt.xlabel("Price (EGP)")
plt.ylabel("Frequency")
plt.show()

brand_stats = df_clean.groupby("Brand")["Price_EGP"].agg(
    ["count", "min", "median", "mean", "max"]
).sort_values("median")

brand_stats["median"].plot(kind="bar", figsize=(10, 5))
plt.title("Median Price by Brand")
plt.xlabel("Brand")
plt.ylabel("Median Price (EGP)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

grade_stats = df_clean.groupby("Grade")["Price_EGP"].agg(
    ["count", "min", "median", "mean", "max"]
)

floor_type_stats = df_clean.groupby("Floor_Type")["Price_EGP"].agg(
    ["count", "min", "median", "mean", "max"]
)


## 8. Outlier Analysis

In [ ]:
q1 = df_clean["Price_EGP"].quantile(0.25)
q3 = df_clean["Price_EGP"].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

outliers = df_clean[
    (df_clean["Price_EGP"] < lower) |
    (df_clean["Price_EGP"] > upper)
].copy()

plt.figure(figsize=(10, 4))
plt.boxplot(df_clean["Price_EGP"].dropna(), vert=False)
plt.title("Price Outlier Check")
plt.xlabel("Price (EGP)")
plt.show()


## 9. Feature Engineering

In [ ]:
df_model = df_clean.copy()

def parse_size(value):
    if pd.isna(value):
        return pd.Series([np.nan, np.nan])
    s = str(value).lower().replace("cm", "").replace(" ", "")
    parts = s.split("x")
    if len(parts) == 2:
        try:
            return pd.Series([float(parts[0]), float(parts[1])])
        except ValueError:
            pass
    return pd.Series([np.nan, np.nan])

df_model[["Size_Length_cm", "Size_Width_cm"]] = df_model["Size_cm"].apply(parse_size)
df_model["Tile_Area_m2"] = (
    df_model["Size_Length_cm"] * df_model["Size_Width_cm"] / 10000
)

unit_lower = df_model["Unit"].astype("string").str.lower()
df_model["Price_per_m2"] = np.where(
    unit_lower.str.contains("m2|m²|square", regex=True, na=False),
    df_model["Price_EGP"],
    np.nan
)

for col in ["Brand", "Floor_Type", "Grade", "Quality_Level", "Unit"]:
    df_model[col] = df_model[col].astype("string").str.strip()


## 10. Quantity & Cost Logic

In [ ]:
DEFAULT_WASTE_FACTOR = 0.10

def flooring_quantity(area_m2, waste_factor=DEFAULT_WASTE_FACTOR):
    return area_m2 * (1 + waste_factor)

def flooring_cost(area_m2, price_per_m2, waste_factor=DEFAULT_WASTE_FACTOR):
    quantity = flooring_quantity(area_m2, waste_factor)
    return quantity * price_per_m2


## 11. Value-for-Money Score

In [ ]:
quality_map = {
    "Economy": 6.0,
    "Medium": 7.5,
    "High": 9.0,
    "Low": 6.0,
    "First": 8.5,
    "Second": 7.0,
    "Third": 5.5
}

df_model["Quality_Score"] = df_model["Quality_Level"].map(quality_map)
df_model["Quality_Score"] = df_model["Quality_Score"].fillna(
    df_model["Grade"].map(quality_map)
)

median_price = df_model["Price_EGP"].median()
df_model["Price_Efficiency"] = median_price / df_model["Price_EGP"].replace(0, np.nan)
df_model["Value_for_Money_Score"] = (
    0.6 * df_model["Quality_Score"] +
    0.4 * (df_model["Price_Efficiency"].clip(0, 2) / 2 * 10)
).round(2)


## 12. Budget-Based Recommendation Function

In [ ]:
def recommend_flooring(area_m2, budget, quality_level=None, floor_type=None,
                        waste_factor=DEFAULT_WASTE_FACTOR, top_n=10):
    data = df_model.copy()

    if floor_type is not None:
        data = data[data["Floor_Type"].str.lower() == str(floor_type).lower()]

    if quality_level is not None:
        data = data[data["Quality_Level"].str.lower() == str(quality_level).lower()]

    data = data[data["Price_EGP"] > 0].copy()
    data["Required_Area_m2"] = area_m2 * (1 + waste_factor)
    data["Estimated_Cost_EGP"] = data["Required_Area_m2"] * data["Price_EGP"]
    data["Within_Budget"] = data["Estimated_Cost_EGP"] <= budget

    return data.sort_values(
        ["Within_Budget", "Value_for_Money_Score"],
        ascending=[False, False]
    ).head(top_n)


## 13. ML Target Definition

In [ ]:
target = "Price_EGP"

feature_candidates = [
    "Brand", "Floor_Type", "Grade", "Size_cm", "Unit",
    "Size_Length_cm", "Size_Width_cm", "Tile_Area_m2"
]

features = [c for c in feature_candidates if c in df_model.columns]

leakage_keywords = [
    "Price_per_m2",
    "Price_Efficiency",
    "Value_for_Money_Score",
    "Quality_Score"
]

features = [
    c for c in features
    if c not in leakage_keywords and c != target
]


## 14. Prepare ML Data

In [ ]:
ml_df = df_model[features + [target]].dropna().copy()

X = ml_df[features]
y = ml_df[target]

categorical_features = X.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

numeric_features = X.select_dtypes(
    include=[np.number]
).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)


## 15. Baseline Price Model

In [ ]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            min_samples_leaf=2,
            n_jobs=-1
        ))
    ]
)

model.fit(X_train, y_train)
pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)


## 16. Actual vs Predicted Price

In [ ]:
results = pd.DataFrame({
    "Actual_Price_EGP": y_test.values,
    "Predicted_Price_EGP": pred
})

plt.figure(figsize=(7, 6))
plt.scatter(
    results["Actual_Price_EGP"],
    results["Predicted_Price_EGP"],
    alpha=0.6
)

min_v = min(results.min())
max_v = max(results.max())

plt.plot([min_v, max_v], [min_v, max_v])
plt.title("Actual vs Predicted Flooring Price")
plt.xlabel("Actual Price (EGP)")
plt.ylabel("Predicted Price (EGP)")
plt.show()


## 17. Final Model-Ready Validation

In [ ]:
required_for_model = features + [target]

checks = {
    "No missing target": ml_df[target].notna().all(),
    "No non-positive target": (ml_df[target] > 0).all(),
    "No duplicate rows": not ml_df.duplicated().any(),
    "Features exist": all(c in df_model.columns for c in features),
    "No target in features": target not in features,
    "No obvious price-derived feature in features": not any(
        c in features for c in leakage_keywords
    )
}

validation_results = pd.DataFrame(
    {"Check": list(checks.keys()), "Passed": list(checks.values())}
)


## 18. Export Prepared Dataset

In [ ]:
final_dataset_path = "flooring_model_ready.csv"
df_model.to_csv(final_dataset_path, index=False, encoding="utf-8-sig")

ml_dataset_path = "flooring_ml_ready.csv"
ml_df.to_csv(ml_dataset_path, index=False, encoding="utf-8-sig")


## 19. Final Summary

In [ ]:
final_summary = {
    "Original rows": len(df),
    "Clean/model rows": len(df_model),
    "Removed duplicates": len(df) - len(df_clean),
    "Columns in final dataset": len(df_model.columns),
    "Brands": df_model["Brand"].nunique(),
    "Floor types": df_model["Floor_Type"].nunique(),
    "Grades": df_model["Grade"].nunique()
}
